# Lesson 11 - A sequential agent workflow

**Level:** beginner | **Demo time:** 5 minutes

This notebook demonstrates a simple sequence of agents:

```text
user -> orchestrator -> idea agent -> writer agent -> orchestrator -> user
```

The orchestrator delegates the request to a two-agent sequential workflow. When the workers finish, control returns to the orchestrator, which returns the final result.

In [ ]:
from agent_framework import Agent
from agent_framework.orchestrations import SequentialBuilder

from maf_foundry_hackathon.config import (
    WorkshopSettings,
    create_chat_client,
    create_credential,
)

settings = WorkshopSettings.from_env()
credential = create_credential()
chat_client = create_chat_client(settings, credential)

## Create the agents

Each agent has one small responsibility. The orchestrator delegates work; the idea and writer agents run sequentially.

In [ ]:
idea_agent = Agent(
    client=chat_client,
    name="idea_agent",
    instructions="Read the user's request and provide three short points the writer should include.",
)

writer_agent = Agent(
    client=chat_client,
    name="writer_agent",
    instructions="Use the user's request and the idea agent's points to write the requested final response.",
)

## Build the worker workflow

`SequentialBuilder` passes the idea agent's output to the writer agent. `as_agent()` makes that complete worker workflow available for delegation.

In [ ]:
worker_workflow = SequentialBuilder(
    participants=[idea_agent, writer_agent]
).build()

writing_team = worker_workflow.as_agent(
    name="writing_team",
    description="Develops ideas and writes a finished response for a user request.",
)

## Create and run the orchestrator

The worker workflow becomes a tool for the orchestrator. The orchestrator calls it, receives its final output, and then answers the user. The application makes only one call.

In [ ]:
orchestrator_agent = Agent(
    client=chat_client,
    name="orchestrator",
    instructions=(
        "Delegate every writing request to the writing_team tool. "
        "After the tool returns, respond with the finished content itself. "
        "Never respond with only an acknowledgement such as 'Done.'"
    ),
    tools=[writing_team.as_tool()],
)

request = (
    "Create a two-sentence launch announcement for Contoso Trail Shoes "
    "that emphasizes comfort and durability."
)

response = await orchestrator_agent.run(request)
print(response.text)

## Recap

One orchestrator call runs the complete sequence:

1. The orchestrator calls the writing-team tool.
2. The idea agent produces input for the writer.
3. The writer produces the requested content.
4. Tool control returns to the original orchestrator.
5. The orchestrator returns the final answer.

A plain sequential workflow only moves forward, so it cannot reuse its first participant at the end. Wrapping the worker workflow as a tool creates a true delegation-and-return pattern without duplicating the orchestrator.

In [ ]:
credential.close()